# Exercise 4: Multiple-choice evaluation

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Pretrained Language Models · version 0.1.0*

> Evaluate Kazakh multiple-choice items with a conditional scorer, as in the Evaluation and Benchmarks Hands-On, and compare it with an encoder whose multiple-choice head is randomly initialised. The latter is not a zero-shot reasoning model.

**Setup.** `pip install torch transformers peft scikit-learn psutil requests "turkicnlp==0.5.0"`. Models are loaded at the Hugging Face revisions pinned in `MODELS` below, from a local Hugging Face cache (`HF_HUB_DIR`, default `~/.cache/huggingface/hub`), from a folder of copies (`MODELS_DIR`) or from the Hub. Data come from commit-pinned GitHub repositories: SIB-200 topic classification (seven topics; FLORES-200 sentences, so the same 1,004 sentences exist in every language), Kardeş-NLU's Kazakh XCOPA and the BERTurk repository's published evaluations. Everything runs on the CPU. Training results are cached per configuration in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes; record elapsed time and thread count for each run.

The items are the Kazakh XCOPA translations of Kardeş-NLU: a premise, a question (cause or effect) and two alternatives. Part (a) runs Glot500 with an untrained multiple-choice head. Part (b) scores each alternative with Qwen2.5-1.5B-Instruct as a continuation of the premise and a connective, and selects the scoring variant on the development items.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, threading, time
from pathlib import Path
import requests

SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KARDES = "https://raw.githubusercontent.com/lksenel/Kardes-NLU/48eea77fda904e731d228de833f4c57d130fad9c/Data/"
BERTURK = "https://raw.githubusercontent.com/stefan-it/turkish-bert/0ab49e288d07f0aa40bcc1c28b655c80c1350fee/"
MODELS = {   # Hugging Face model -> pinned revision
    "cis-lmu/glot500-base": "d4d7c1ec01828fdf7452a4ccf7b55177aced175e",
    "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a",
    "meta-llama/Meta-Llama-3-8B": "8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920",
    "allenai/Olmo-3-1025-7B": "0a15c289b17ad4f27aabc863276ec525d5312c01",
    "Qwen/Qwen3.6-27B": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
HF_HUB_DIR = Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", CACHE / "models"))
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def model_source(name):
    """(path or name, revision) for from_pretrained: a local snapshot of the pinned revision if present."""
    rev = MODELS[name]
    for p in (HF_HUB_DIR / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              MODELS_DIR / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev

def tokenizer(name):
    from transformers import AutoTokenizer
    path, rev = model_source(name)
    return AutoTokenizer.from_pretrained(path, revision=rev)

def sib(lang, split):
    """SIB-200 rows (index_id, topic, text) for a FLORES language code such as kaz_Cyrl."""
    text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
    return [(r["index_id"], r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]

LABELS = ["entertainment", "geography", "health", "politics", "science/technology", "sports", "travel"]
GLOT = "cis-lmu/glot500-base"

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Alphabet version of a sentence (TurkicNLP transliterator); Turkish is unchanged."""
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = {"tur": Script.LATIN}.get(lang, Script.CYRILLIC)
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def macro_f1(gold, pred):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)

class PeakRSS:
    """Peak resident memory of this process while the block runs (sampled every 0.2 s)."""
    def __enter__(self):
        import psutil
        self.p, self.peak, self.stop = psutil.Process(), 0, False
        def sample():
            while not self.stop:
                self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(0.2)
        self.t = threading.Thread(target=sample, daemon=True); self.t.start(); return self
    def __exit__(self, *exc):
        self.stop = True; self.t.join()

def run(mode="lora", r=8, train="kaz_Cyrl", script="native", seed=1, epochs=10, new_tokens=0,
        tests=(("kaz_Cyrl", "native"),)):
    """Fine-tune Glot500-base for SIB-200 topics; select the epoch on the training language's dev
    macro-F1; return test macro-F1/accuracy for each (language, script) in `tests`. Cached on disk.
    mode: "head" (classifier only), "lora" (query/value, rank r, alpha 2r, classifier trained),
    "full" (all weights except the word embeddings). new_tokens > 0 adds that many Kazakh words to
    the tokenizer (see add_kazakh_words) and trains their embedding rows together with LoRA."""
    cfg = dict(base=GLOT, rev=MODELS[GLOT], mode=mode, r=r if mode == "lora" else None, train=train,
               script=script, seed=seed, epochs=epochs, new_tokens=new_tokens, tests=[list(t) for t in tests],
               batch=16, lr={"head": 1e-3, "lora": 2e-4, "full": 2e-5}[mode], warmup=0.1, v=1)
    key = hashlib.sha1(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]
    path = CACHE / "ch12-runs" / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text())
    import torch
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    torch.manual_seed(seed); torch.set_num_threads(THREADS)
    src, rev = model_source(GLOT)
    tok = tokenizer(GLOT)
    model = AutoModelForSequenceClassification.from_pretrained(
        src, revision=rev, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)),
        label2id={l: i for i, l in enumerate(LABELS)})
    added = add_kazakh_words(tok, model, new_tokens) if new_tokens else []
    if mode == "full":
        model.roberta.embeddings.word_embeddings.weight.requires_grad_(False)
    elif mode == "head":
        for n, p in model.named_parameters():
            p.requires_grad_(n.startswith("classifier"))
    else:
        from peft import LoraConfig, TaskType, get_peft_model
        extra = {"trainable_token_indices": {"word_embeddings": added}} if added else {}
        model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r, lora_dropout=0.1,
                                                 target_modules=["query", "value"], modules_to_save=["classifier"], **extra))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    def data(lang, split, scr):
        rows = sib(lang, split)
        texts = [t if scr == "native" else to_cts(lang.split("_")[0], t) for _, _, t in rows]
        return texts, [LABELS.index(c) for _, c, _ in rows]
    def predict(texts):
        model.eval(); out = []
        with torch.inference_mode():
            for i in range(0, len(texts), 32):
                enc = tok(texts[i:i + 32], padding=True, truncation=True, max_length=256, return_tensors="pt")
                out += model(**enc).logits.argmax(-1).tolist()
        return out
    tr_x, tr_y = data(train, "train", script)
    dev_x, dev_y = data(train, "dev", script)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.01)
    steps = epochs * math.ceil(len(tr_x) / cfg["batch"])
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup"] * steps), steps)
    rng = random.Random(seed); best = (-1, None, None); history = []
    t0 = time.time()
    with PeakRSS() as mem:
        for ep in range(1, epochs + 1):
            model.train(); order = list(range(len(tr_x))); rng.shuffle(order)
            for i in range(0, len(order), cfg["batch"]):
                b = order[i:i + cfg["batch"]]
                enc = tok([tr_x[j] for j in b], padding=True, truncation=True, max_length=256, return_tensors="pt")
                loss = model(**enc, labels=torch.tensor([tr_y[j] for j in b])).loss
                loss.backward(); opt.step(); sched.step(); opt.zero_grad()
            f1 = macro_f1(dev_y, predict(dev_x)); history.append(round(f1, 4))
            if f1 > best[0]:
                best = (f1, ep, {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad})
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best[2]:
                    p.copy_(best[2][n])
        results = {}
        for lang, scr in tests:
            x, y = data(lang, "test", scr); pred = predict(x)
            results[f"{lang}/{scr}"] = {"macro_f1": macro_f1(y, pred), "accuracy": sum(a == b for a, b in zip(y, pred)) / len(y),
                                        "n": len(y), "pred": pred}
    res = dict(cfg=cfg, trainable=trainable, total=total, dev_history=history, best_epoch=best[1],
               dev_macro_f1=best[0], tests=results, seconds=time.time() - t0, peak_rss_gb=mem.peak / 1e9,
               added_tokens=added)
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(res))
    return res

PROBE = r"""
import json, sys, threading, time, psutil, torch
a = json.loads(sys.argv[1]); torch.set_num_threads(a["threads"]); torch.manual_seed(0)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
proc = psutil.Process(); start = proc.memory_info().rss
tok = AutoTokenizer.from_pretrained(a["path"]); m = AutoModelForSequenceClassification.from_pretrained(a["path"], num_labels=7)
if a["mode"] == "full":
    m.roberta.embeddings.word_embeddings.weight.requires_grad_(False)
elif a["mode"] == "head":
    for n, p in m.named_parameters(): p.requires_grad_(n.startswith("classifier"))
else:
    from peft import LoraConfig, TaskType, get_peft_model
    m = get_peft_model(m, LoraConfig(task_type=TaskType.SEQ_CLS, r=a["r"], lora_alpha=2 * a["r"], lora_dropout=0.1,
                                     target_modules=["query", "value"], modules_to_save=["classifier"]))
loaded = proc.memory_info().rss; peak = [loaded]; stop = []
def sample():
    while not stop: peak[0] = max(peak[0], proc.memory_info().rss); time.sleep(0.05)
t = threading.Thread(target=sample, daemon=True); t.start()
opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-4)
enc = tok(a["texts"], padding=True, return_tensors="pt"); y = torch.zeros(len(a["texts"]), dtype=torch.long)
t0 = time.time()
for _ in range(a["steps"]):
    m(**enc, labels=y).loss.backward(); opt.step(); opt.zero_grad()
sec = (time.time() - t0) / a["steps"]; stop.append(1); t.join()
print(json.dumps({"start": start, "loaded": loaded, "peak": peak[0], "sec_per_step": sec}))
"""

def peak_memory(mode, r=8, steps=5):
    """Peak resident memory of a fresh process that loads Glot500 and takes `steps` training steps
    on the first 16 Kazakh training sentences (batch 16), and its seconds per step. Cached."""
    import subprocess, sys
    key = f"{mode}-{r if mode == 'lora' else ''}-{steps}-{THREADS}"
    path = CACHE / "ch12-memory.json"
    saved = json.loads(path.read_text()) if path.exists() else {}
    if key not in saved:
        texts = [t for _, _, t in sib("kaz_Cyrl", "train")[:16]]
        args = dict(path=model_source(GLOT)[0], mode=mode, r=r, steps=steps, threads=THREADS, texts=texts)
        out = subprocess.run([sys.executable, "-c", PROBE, json.dumps(args, ensure_ascii=False)],
                             capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
        saved[key] = json.loads(out); path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(saved))
    return saved[key]

def kazakh_candidates(tok, n):
    """The n most frequent Kazakh word forms of the SIB-200 training split that Glot500 splits into
    three or more pieces (ties broken alphabetically)."""
    words = collections.Counter(w for _, _, t in sib("kaz_Cyrl", "train") for w in re.findall(r"\w+", t))
    long = [w for w, c in sorted(words.items(), key=lambda x: (-x[1], x[0]))
            if len(tok.tokenize(" " + w)) >= 3]
    return long[:n]

def add_kazakh_words(tok, model, n):
    """Add whole-word tokens without changing old IDs; initialise each new row as the mean of the
    old embeddings of the word's pieces. Returns the new IDs."""
    import torch
    from tokenizers import AddedToken
    words = kazakh_candidates(tok, n)
    old_len = len(tok)
    pieces = {w: tok(" " + w, add_special_tokens=False)["input_ids"] for w in words}
    tok.add_tokens([AddedToken(w, single_word=True, lstrip=True) for w in words])
    model.resize_token_embeddings(len(tok), mean_resizing=False)
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for w in words:
            emb[tok.convert_tokens_to_ids(w)] = emb[pieces[w]].mean(0)
    return list(range(old_len, len(tok)))

def wilson(k, n, z=1.959963984540054):
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

print("models:", ", ".join(f"{n} ({'local' if model_source(n)[1] is None else 'Hub'})" for n in MODELS))

In [ ]:
import torch
torch.set_num_threads(THREADS)
def xcopa(split):
    text = fetch(f"{KARDES}kazakh/xcopa.{split}.kk.csv").read_text(encoding="utf-8-sig")
    return list(csv.DictReader(io.StringIO(text)))
dev, test = xcopa("dev"), xcopa("test")
print(f"Kardeş-NLU Kazakh XCOPA: dev {len(dev)}, test {len(test)}; two alternatives each, so chance = 50%; "
      f"gold = alternative 1 in {sum(r['most-plausible-alternative'] == '1' for r in test)} of {len(test)} test items")

# (a) An encoder with a randomly initialised multiple-choice head, no training.
from transformers import AutoModelForMultipleChoice
src, rev = model_source(GLOT); gtok = tokenizer(GLOT)
for seed in (1, 2, 3):
    torch.manual_seed(seed)
    mc = AutoModelForMultipleChoice.from_pretrained(src, revision=rev).eval()
    correct = 0
    with torch.inference_mode():
        for r in test:
            enc = gtok([r["p_translation"]] * 2, [r["a1_translation"], r["a2_translation"]], padding=True, return_tensors="pt")
            pred = mc(**{k: v.unsqueeze(0) for k, v in enc.items()}).logits.argmax(-1).item() + 1
            correct += pred == int(r["most-plausible-alternative"])
    lo, hi = wilson(correct, len(test))
    print(f"Glot500 + random multiple-choice head, seed {seed}: {correct}/{len(test)} = {correct / len(test):.1%} (95% CI {lo:.1%}-{hi:.1%})")
    del mc

# (b) Conditional scoring with a generative model: log P(alternative | premise + connective).
from transformers import AutoModelForCausalLM
src, rev = model_source("Qwen/Qwen2.5-1.5B-Instruct")
qtok = tokenizer("Qwen/Qwen2.5-1.5B-Instruct")
lm = AutoModelForCausalLM.from_pretrained(src, revision=rev, dtype=torch.bfloat16).eval()
CONNECTIVE = {"kk": {"cause": "себебі", "effect": "сондықтан"}, "en": {"cause": "because", "effect": "so"}}
SCORES = CACHE / "ch12-xcopa-scores.json"
scores = json.loads(SCORES.read_text()) if SCORES.exists() else {}
def score(prefix, cont):
    """Sum and count of log-probabilities of the continuation tokens."""
    key = prefix + "\t" + cont
    if key not in scores:
        a = qtok(prefix, add_special_tokens=False)["input_ids"]; b = qtok(cont, add_special_tokens=False)["input_ids"]
        with torch.inference_mode():
            logp = lm(torch.tensor([a + b])).logits[0, :-1].float().log_softmax(-1)
        scores[key] = [sum(logp[len(a) - 1 + i, t].item() for i, t in enumerate(b)), len(b)]
    return scores[key]
def item(r, lang):
    p, a1, a2 = ((r["p_translation"], r["a1_translation"], r["a2_translation"]) if lang == "kk" else (r["p"], r["a1"], r["a2"]))
    prefix = p.rstrip(".") + ", " + CONNECTIVE[lang][r["asks-for"]]
    return [score(prefix, " " + a[0].lower() + a[1:]) for a in (a1, a2)]
def accuracy(rows, lang, norm):
    ok = 0
    for r in rows:
        s = [t / n if norm else t for t, n in item(r, lang)]
        ok += (1 if s[0] >= s[1] else 2) == int(r["most-plausible-alternative"])
    return ok
for lang in ("kk", "en"):
    devacc = {norm: accuracy(dev, lang, norm) for norm in (False, True)}
    choice = max(devacc, key=devacc.get)
    k = accuracy(test, lang, choice); lo, hi = wilson(k, len(test))
    print(f"Qwen2.5-1.5B-Instruct, {'Kazakh' if lang == 'kk' else 'English original'}: dev sum {devacc[False]}/{len(dev)}, "
          f"mean {devacc[True]}/{len(dev)} -> {'mean' if choice else 'sum'}; test {k}/{len(test)} = {k / len(test):.1%} (95% CI {lo:.1%}-{hi:.1%})")
SCORES.parent.mkdir(parents=True, exist_ok=True); SCORES.write_text(json.dumps(scores))
del lm                                                # free the 1.5B model before other cells
r = test[0]; print("prefix example:", r["p_translation"].rstrip(".") + ", " + CONNECTIVE["kk"][r["asks-for"]], "|", r["a1_translation"], "/", r["a2_translation"])

## Worked solution

Output of `experiments/ch12_exercises/run_ch12_exercises.py`:

```
Kardeş-NLU Kazakh XCOPA: dev 100, test 500; two alternatives each, so chance = 50%; gold = alternative 1 in 250 of 500 test items
Glot500 + random multiple-choice head, seed 1: 262/500 = 52.4% (95% CI 48.0%-56.7%)
Glot500 + random multiple-choice head, seed 2: 232/500 = 46.4% (95% CI 42.1%-50.8%)
Glot500 + random multiple-choice head, seed 3: 214/500 = 42.8% (95% CI 38.5%-47.2%)
Qwen2.5-1.5B-Instruct, Kazakh: dev sum 55/100, mean 50/100 -> sum; test 261/500 = 52.2% (95% CI 47.8%-56.5%)
Qwen2.5-1.5B-Instruct, English original: dev sum 82/100, mean 82/100 -> sum; test 421/500 = 84.2% (95% CI 80.7%-87.1%)
prefix example: Элемент көпіршікті орамамен оралған, себебі | Ол сыңғыш болды. / Ол кішкентай болды.
```

**An untrained head.** Glot500 with a randomly initialised multiple-choice head scores 52.4%, 46.4% and 42.8% across three seeds. These results do not evaluate a trained multiple-choice system. A random head is not equivalent to independent uniform guessing, so its score need not be near 50%.

**Conditional scoring.** Qwen2.5-1.5B-Instruct scores 261/500 (52.2%) on Kazakh, with a 95% interval of 47.8–56.5% that includes the 50% random-choice baseline. The English originals score 421/500 (84.2%). This shows a gap for the tested model, prompts and scoring rule. It does not separate language proficiency, reasoning, translation quality or possible prior exposure to benchmark items.

**Protocol.** Sum and mean token log-probability were compared on 100 development items; the sum was selected for both languages. The test labels are balanced, so always choosing the same alternative scores 50%. The Kazakh items are translated COPA questions, not a test of Kazakhstan-specific knowledge. Choose new prompt variants on development data rather than retuning on these test results.

## Your turn — extension

1. Translate the connectives differently (for example *өйткені* for 'because') and measure how much the Kazakh score moves.
2. Score the Kazakh items with the Kazakh-adapted KAZ-LLM or Sherkala if you can run an 8-billion-parameter model.